# Colab GPU Training

Use **Runtime > Change runtime type > GPU** before running this notebook. Upload `outputs/colab/insurance_capstone_colab_bundle.zip` when prompted. The notebook installs optional GPU-capable model packages, runs `python run_all.py` with `INSURANCE_USE_GPU=1`, then downloads the regenerated artifacts.

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q pandas numpy scikit-learn lightgbm xgboost catboost shap matplotlib seaborn joblib python-docx python-pptx pypdf nbformat streamlit

In [ ]:
from google.colab import files
from pathlib import Path
import os
import shutil
import zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_names, 'Upload outputs/colab/insurance_capstone_colab_bundle.zip'

work_root = Path('/content/insurance_capstone')
if work_root.exists():
    shutil.rmtree(work_root)
work_root.mkdir(parents=True)

with zipfile.ZipFile(zip_names[0]) as archive:
    archive.extractall(work_root)

candidates = sorted(work_root.rglob('run_all.py'))
assert candidates, 'run_all.py was not found in the uploaded project bundle'
project_root = candidates[0].parent
os.chdir(project_root)
print(f'Project root: {project_root}')

In [ ]:
import os

os.environ['INSURANCE_USE_GPU'] = '1'
!python run_all.py

In [ ]:
import json
import pandas as pd
from pathlib import Path

metrics = pd.read_csv('outputs/models/model_metrics.csv')
display(metrics.head(12)[['model', 'target_transform', 'test_MAE', 'test_RMSE', 'test_R2', 'cv_rmse_mean']])

metadata = json.loads(Path('outputs/models/model_metadata.json').read_text())
print(json.dumps({
    'final_model': metadata.get('final_model'),
    'test_mae': metadata.get('test_mae'),
    'test_rmse': metadata.get('test_rmse'),
    'test_r2': metadata.get('test_r2'),
    'external_model_packages': metadata.get('external_model_packages'),
    'gpu_acceleration_enabled': metadata.get('gpu_acceleration_enabled'),
    'gpu_model_candidates': metadata.get('gpu_model_candidates'),
}, indent=2))

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

artifact_zip = Path('/content/insurance_colab_artifacts.zip')
paths_to_pack = [
    'outputs/models',
    'outputs/tables',
    'outputs/figures',
    'outputs/reports',
    'Insurance_Price_Prediction_Project_Report.html',
    'Milestone_1_Insurance_Price_Prediction.docx',
    'Milestone_2_Insurance_Price_Prediction.docx',
    'Insurance_Price_Prediction_Final_Presentation.pptx',
    'README.md',
    'requirements.txt',
    'app.py',
    'insurance_modeling.py',
    'run_all.py',
]

with zipfile.ZipFile(artifact_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for item in paths_to_pack:
        path = project_root / item
        if path.is_dir():
            for file_path in path.rglob('*'):
                if file_path.is_file() and '__pycache__' not in file_path.parts:
                    archive.write(file_path, file_path.relative_to(project_root))
        elif path.exists():
            archive.write(path, path.relative_to(project_root))

files.download(str(artifact_zip))